# Lab 22: Text as Vectors

## How words, documents, and meaning become geometry

This lab is a deeper computational companion to Chapter 22. We will build text vectors from scratch, compare documents, construct TF-IDF weights, use SVD to find hidden topics, and end with a high-dimensional simulation that explains why text data is both powerful and difficult.

You should read this notebook as a guided story. Each section combines writing, computation, visualization, and short reflection questions.

## 1. Setup

We use only standard Python tools plus NumPy, pandas, and matplotlib. The point is not to hide the ideas inside a library. The point is to see the linear algebra.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 2. A small text collection

We begin with a small artificial collection. It has three rough themes: mathematics/AI, music, and sports.

In [ ]:
documents = [
    "linear algebra data matrix vector model ai",
    "neural network model data ai learning",
    "matrix factorization vector data model",
    "guitar melody song music rhythm",
    "music guitar piano melody song",
    "rhythm melody concert music guitar",
    "basketball team score player game",
    "soccer team player goal score game",
    "game coach team score player",
]
labels = ["AI", "AI", "AI", "Music", "Music", "Music", "Sports", "Sports", "Sports"]

pd.DataFrame({"label": labels, "document": documents})

## 3. Tokenization and vocabulary

A vocabulary is a coordinate system. Each word becomes one axis.

We will use a very simple tokenizer: lowercase the text and extract word tokens.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

tokens = [tokenize(doc) for doc in documents]
vocab = sorted(set(word for doc in tokens for word in doc))
word_to_index = {w:i for i,w in enumerate(vocab)}

print("Vocabulary size:", len(vocab))
print(vocab)

## 4. Build the document-term matrix from scratch

The document-term matrix $X$ has one row per document and one column per word.

The entry $X_{ij}$ is the number of times word $j$ appears in document $i$.

In [ ]:
X = np.zeros((len(documents), len(vocab)), dtype=float)
for i, doc_tokens in enumerate(tokens):
    for word in doc_tokens:
        X[i, word_to_index[word]] += 1

X_df = pd.DataFrame(X, columns=vocab, index=[f"D{i+1}" for i in range(len(documents))])
X_df

### Reflection

Look at the matrix above. Which columns are active for the AI documents? Which are active for the music documents? Which are active for the sports documents?

## 5. Visualizing a document-term matrix

A document-term matrix is an image: rows are documents, columns are words, and brightness represents word count.

In [ ]:
plt.figure(figsize=(10,4))
plt.imshow(X, aspect='auto')
plt.xticks(range(len(vocab)), vocab, rotation=70, ha='right')
plt.yticks(range(len(documents)), [f"D{i+1} ({labels[i]})" for i in range(len(documents))])
plt.colorbar(label="count")
plt.title("Document-term matrix as an image")
plt.tight_layout()
plt.show()

## 6. Cosine similarity

For text, cosine similarity is often more useful than raw dot product because it compares direction rather than document length.

In [ ]:
def cosine(x, y):
    nx = np.linalg.norm(x)
    ny = np.linalg.norm(y)
    if nx == 0 or ny == 0:
        return 0.0
    return float(x @ y / (nx * ny))

S = np.array([[cosine(X[i], X[j]) for j in range(len(documents))] for i in range(len(documents))])
S_df = pd.DataFrame(S, index=[f"D{i+1}" for i in range(len(documents))], columns=[f"D{i+1}" for i in range(len(documents))])
S_df.round(2)

In [ ]:
plt.figure(figsize=(6,5))
plt.imshow(S, vmin=0, vmax=1)
plt.xticks(range(len(documents)), [f"D{i+1}" for i in range(len(documents))])
plt.yticks(range(len(documents)), [f"D{i+1}" for i in range(len(documents))])
plt.colorbar(label="cosine similarity")
plt.title("Document similarity matrix")
plt.tight_layout()
plt.show()

## 7. A tiny search engine

A query is also a vector. We vectorize the query using the same vocabulary, then compare it with every document.

In [ ]:
def vectorize(text, vocab=vocab, word_to_index=word_to_index):
    v = np.zeros(len(vocab), dtype=float)
    for word in tokenize(text):
        if word in word_to_index:
            v[word_to_index[word]] += 1
    return v

def search(query, matrix=X, top=5):
    q = vectorize(query)
    scores = np.array([cosine(q, matrix[i]) for i in range(matrix.shape[0])])
    order = np.argsort(scores)[::-1]
    return pd.DataFrame({
        "rank": np.arange(1, top+1),
        "document": [documents[i] for i in order[:top]],
        "label": [labels[i] for i in order[:top]],
        "score": scores[order[:top]]
    })

search("matrix data ai model", top=5)

In [ ]:
search("guitar melody concert", top=5)

In [ ]:
search("team game score", top=5)

### Student task

Try your own query below. Predict the top result before running the code.

In [ ]:
search("your query words here", top=5)

## 8. TF-IDF weighting

Raw word counts are not always ideal. Words that appear in many documents should often be less important than words that distinguish a document.

TF-IDF combines local frequency and global rarity.

In [ ]:
# Term frequency
row_sums = X.sum(axis=1, keepdims=True)
tf = X / row_sums

# Smooth inverse document frequency
N = X.shape[0]
df = (X > 0).sum(axis=0)
idf = np.log((N + 1) / (df + 1)) + 1

X_tfidf = tf * idf
pd.DataFrame(X_tfidf, columns=vocab, index=[f"D{i+1}" for i in range(len(documents))]).round(3)

In [ ]:
idf_df = pd.DataFrame({"term": vocab, "document_frequency": df, "idf": idf}).sort_values("idf", ascending=False)
idf_df

## 9. Compare search with raw counts and TF-IDF

The same query may rank documents differently after TF-IDF weighting.

In [ ]:
def vectorize_tfidf_query(text):
    q_counts = vectorize(text)
    if q_counts.sum() == 0:
        return q_counts
    q_tf = q_counts / q_counts.sum()
    return q_tf * idf

def search_tfidf(query, top=5):
    q = vectorize_tfidf_query(query)
    scores = np.array([cosine(q, X_tfidf[i]) for i in range(X_tfidf.shape[0])])
    order = np.argsort(scores)[::-1]
    return pd.DataFrame({
        "rank": np.arange(1, top+1),
        "document": [documents[i] for i in order[:top]],
        "label": [labels[i] for i in order[:top]],
        "tfidf_score": scores[order[:top]]
    })

query = "data model matrix"
print("Raw count search")
display(search(query, top=5))
print("TF-IDF search")
display(search_tfidf(query, top=5))

## 10. SVD and hidden topics

The document-term matrix may contain hidden low-dimensional structure. SVD decomposes the matrix into dominant patterns.

If $X=U\Sigma V^T$, then the rows of $U\Sigma$ are document coordinates in a low-dimensional topic space, and the rows of $V^T$ describe word patterns.

In [ ]:
U, s, Vt = np.linalg.svd(X_tfidf, full_matrices=False)
print("Singular values:", np.round(s, 3))

plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(s)+1), s, marker='o')
plt.xlabel("component")
plt.ylabel("singular value")
plt.title("Scree plot for text matrix")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
for k in range(3):
    weights = Vt[k]
    top_idx = np.argsort(np.abs(weights))[::-1][:6]
    print(f"Component {k+1} top terms:")
    for idx in top_idx:
        print(f"  {vocab[idx]:>12s}: {weights[idx]: .3f}")
    print()

## 11. Plot documents in a 2D SVD topic space

We can project each document into the first two SVD coordinates.

In [ ]:
coords = U[:, :2] * s[:2]
plt.figure(figsize=(7,6))
for lab in sorted(set(labels)):
    idx = [i for i,l in enumerate(labels) if l == lab]
    plt.scatter(coords[idx,0], coords[idx,1], label=lab, s=80)
for i, (x0,y0) in enumerate(coords):
    plt.text(x0+0.005, y0+0.005, f"D{i+1}")
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel("SVD coordinate 1")
plt.ylabel("SVD coordinate 2")
plt.title("Documents in a low-dimensional topic space")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 12. Low-rank reconstruction

A rank-$k$ approximation keeps only the top $k$ topic directions. This is a compressed version of the text matrix.

In [ ]:
def rank_k_approx(U, s, Vt, k):
    return (U[:, :k] * s[:k]) @ Vt[:k, :]

errors = []
for k in range(1, min(X_tfidf.shape)+1):
    Xk = rank_k_approx(U, s, Vt, k)
    errors.append(np.linalg.norm(X_tfidf - Xk, 'fro'))

plt.figure(figsize=(6,4))
plt.plot(range(1, len(errors)+1), errors, marker='o')
plt.xlabel("rank k")
plt.ylabel("reconstruction error")
plt.title("Low-rank approximation error")
plt.grid(True, alpha=0.3)
plt.show()

## 13. High-dimensional sparse text simulation

Real text vectors are high-dimensional and sparse. Let us simulate documents in a vocabulary of size $5000$, where each document uses only $40$ words.

In [ ]:
rng = np.random.default_rng(7)
num_docs = 200
vocab_size = 5000
words_per_doc = 40

X_sparse = np.zeros((num_docs, vocab_size), dtype=float)
for i in range(num_docs):
    idx = rng.choice(vocab_size, size=words_per_doc, replace=False)
    X_sparse[i, idx] = rng.integers(1, 4, size=words_per_doc)

fraction_zero = np.mean(X_sparse == 0)
print("Shape:", X_sparse.shape)
print("Fraction of zero entries:", fraction_zero)

## 14. Cosine similarities in high dimension

In a huge vocabulary, random sparse documents often have almost no word overlap. This creates challenges for basic bag-of-words comparison.

In [ ]:
pairs = rng.integers(0, num_docs, size=(1000, 2))
cosines = []
for i,j in pairs:
    if i != j:
        cosines.append(cosine(X_sparse[i], X_sparse[j]))

plt.figure(figsize=(6,4))
plt.hist(cosines, bins=30)
plt.xlabel("cosine similarity")
plt.ylabel("number of pairs")
plt.title("Cosine similarities of random sparse documents")
plt.grid(True, alpha=0.3)
plt.show()

print("Mean cosine:", np.mean(cosines))
print("Max cosine:", np.max(cosines))

## 15. Why embeddings matter

Bag-of-words vectors only know exact word overlap. If one document says "car" and another says "automobile," their count vectors may look unrelated.

Embeddings try to solve this problem by learning dense vectors where semantic neighbors are geometrically close.

In this final toy example, we manually create tiny two-dimensional embeddings to show the idea.

In [ ]:
emb = {
    "car": np.array([1.0, 0.1]),
    "automobile": np.array([0.95, 0.15]),
    "truck": np.array([0.9, 0.25]),
    "guitar": np.array([-0.2, 1.0]),
    "piano": np.array([-0.1, 0.95]),
    "music": np.array([-0.15, 0.9]),
}

plt.figure(figsize=(6,5))
for word, vec in emb.items():
    plt.scatter(vec[0], vec[1], s=80)
    plt.text(vec[0]+0.02, vec[1]+0.02, word)
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.title("Toy word embeddings")
plt.xlabel("embedding coordinate 1")
plt.ylabel("embedding coordinate 2")
plt.grid(True, alpha=0.3)
plt.show()

## Final reflection

Write short answers to these questions:

1. What does a vocabulary do geometrically?
2. Why is cosine similarity useful for text?
3. What does TF-IDF change about the geometry?
4. Why can SVD reveal topics?
5. What do embeddings add beyond bag-of-words?
6. What is one limitation of each representation: counts, TF-IDF, SVD topics, and embeddings?